# Test for ragchecker

In [ ]:
import os

from dotenv import load_dotenv
load_dotenv()

import nest_asyncio
nest_asyncio.apply()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from ragchecker.integrations.llama_index import response_to_rag_results
from ragchecker import RAGResults, RAGChecker
from ragchecker.metrics import all_metrics

### Create a naive model

In [ ]:
! pip install llama-index-readers-web

In [ ]:
from llama_index.readers.web import SimpleWebPageReader
documents = SimpleWebPageReader(html_to_text=True).load_data(
    ["http://www.ml.school/"]
)

In [ ]:
from llama_index.core import SummaryIndex
index = SummaryIndex.from_documents(documents)

In [ ]:
# Create query engine
rag_application = index.as_query_engine()

In [ ]:
# User query and groud truth answer
user_query = "What is Price of this course?"
gt_answer = "The price of the course is $500"

In [ ]:
# Get response from LlamaIndex
response_object = rag_application.query(user_query)

In [ ]:
response_object

In [ ]:
# Convert to RAGChecker format
rag_result = response_to_rag_results(
    query=user_query,
    gt_answer=gt_answer,
    response_object=response_object,
)

In [ ]:
rag_result

In [ ]:
if rag_result.get('query_id') is None:
    rag_result['query_id'] = "001"

In [ ]:
rag_result

In [ ]:
# Create RAGResults object
rag_results = RAGResults.from_dict({"results": [rag_result]})

In [ ]:
rag_results

In [ ]:
with open("/Users/chris/Desktop/7980/CS7980_Capstone_RBCMuseum/eval/rag_response.json") as fp:
    rag_result2 = fp.read()

In [ ]:
rag_result2

In [ ]:
rag_results2 = RAGResults.from_json(rag_result2)

In [ ]:
rag_results2

In [ ]:
import os
from refchecker import LLMExtractor, LLMChecker


os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# GPT-4o from OpenAI
model = 'gpt-4o'

extractor = LLMExtractor(model=model, batch_size=8)
checker = LLMChecker(model=model, batch_size=8)

In [ ]:
import litellm

# Set the API key for litellm (OpenAI API)
litellm.api_key = OPENAI_API_KEY


# Initialize RAGChecker
# Ensure no `max_workers` is being passed or used internally
evaluator = RAGChecker(
    extractor_name=extractor,
    checker_name=checker,
    batch_size_extractor=32,
    batch_size_checker=32
)

# Evaluate using RAGChecker
evaluator.evaluate(rag_results2, all_metrics)

# Print detailed results
print(rag_results)

In [ ]:
from litellm import completion

model = "gpt-4-1106-preview"  # or "gpt-4-0125-preview" for the latest version

# Initialize RAGChecker
evaluator = RAGChecker(
    extractor=lambda prompt: completion(model=model, messages=[{"role": "user", "content": prompt}]),
    checker=lambda prompt: completion(model=model, messages=[{"role": "user", "content": prompt}]),
    batch_size_extractor=32,
    batch_size_checker=32,
)

# Evaluate using RAGChecker
evaluator.evaluate(rag_results, all_metrics)

# Print detailed results
print(rag_results)

### It is confirmed that litellm worked with the OPENAI API KEY

In [ ]:
import os 
from litellm import completion

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# openai call
response = completion(
    model = "gpt-4o", 
    messages=[{ "content": "Hello, how are you?","role": "user"}]
)

In [ ]:
response